## Rule-based data integration pipeline (Standard Blocker)

In [2]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
from PyDI.io import load_parquet

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [4]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher


# standard blocker: metabooks > amazon
st_blocker_m2a = StandardBlocker(
    metabooks, amazon_books,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

# standard blocker: metabooks > goodreads
st_blocker_m2g = StandardBlocker(
    metabooks, goodreads,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
comparators_m2a = [
    StringComparator(column="title_norm", similarity_function="cosine"),
    StringComparator(column="author_norm", similarity_function="cosine", preprocess=str.lower),
    StringComparator(column="publisher", similarity_function="cosine", preprocess=str.lower),
    StringComparator(column="isbn_clean", similarity_function="cosine"),
    NumericComparator(column="publish_year", max_difference=1),
]

comparators_m2g = comparators_m2a + [
    StringComparator(column="genres", similarity_function="jaccard",
                     preprocess=str.lower, list_strategy="concatenate"),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2),
]

In [6]:
matcher = RuleBasedMatcher()

# matching metabooks > amazon
st_corr_m2a = matcher.match(
    df_left=metabooks,
    df_right=amazon_books, 
    candidates=st_blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.4, 0.2, 0.1, 0.2, 0.1], 
    threshold=0.7,
    id_column='id'
)

# matching metabooks > goodreads
st_corr_m2g = matcher.match(
    df_left=metabooks,
    df_right=goodreads, 
    candidates=st_blocker_m2g,
    comparators=comparators_m2g,
    weights=[0.4, 0.25, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05], 
    threshold=0.7,
    id_column='id'
)

In [7]:
# st_corr_m2a.to_csv(CORR_DIR / "st_corr_m2a.csv", index=False)
# st_corr_m2g.to_csv(CORR_DIR / "st_corr_m2g.csv", index=False)

In [8]:
# data fusion for standard blocker
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

all_standard_correspondences = pd.concat([st_corr_m2a, st_corr_m2g], ignore_index=True)

len(all_standard_correspondences)

20134

In [9]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title_norm', longest_string)
strategy.add_attribute_fuser('author_norm', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [10]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_standard_blocker.jsonl")

fused_standard_blocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_standard_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_standard_blocker):,}')
display(fused_standard_blocker.head())

Fused rows: 17,637


,_id,_fusion_group_id,_fusion_sources,title,price,page_count,publisher,id,author_norm,publish_year,...,rating_number,title_norm,language,rating,genres,isbn_clean,_fusion_confidence,_fusion_metadata,edition,book_format
0,metabooks_340343,group_0,"[metabooks, amazon_books]",Dirt,87.959999,362.0,Uglytown Productions,metabooks_340343,sean doolittle,2001.0,...,6,dirt,English,4.4,"[['Literature', 'Fiction', 'United States']]",096634734X,0.769231,"{'title_rule': 'first_non_null', 'title_inputs...",NaN,NaN
1,metabooks_498722,group_1,"[metabooks, amazon_books, goodreads]","They Shoot Canoes, Don't They?",6.900000,240.0,Owl Books (NY),metabooks_498722,patrick f mcmanus,1982.0,...,616,they shoot canoes dont they,English,4.7,"[['Humor', 'Comedy', 'Fiction', 'Outdoors', 'S...",0805000305,0.666667,"{'title_rule': 'first_non_null', 'title_inputs...",nan,Paperback
2,metabooks_438873,group_2,"[metabooks, amazon_books]",Stone Butch Blues: A Novel,80.000000,320.0,Consortium,metabooks_438873,leslie feinberg,2004.0,...,231,stone butch blues a novel,English,4.7,"[['LGBTQ+ Books', 'Literature', 'Fiction']]",1555838537,0.769231,"{'title_rule': 'first_non_null', 'title_inputs...",NaN,NaN
3,amazon_24285,group_3,"[metabooks, amazon_books]",The Path : Creating Your Mission Statement for...,7.960000,224.0,Hyperion,amazon_24285,laurie beth jones,1998.0,...,284,the path creating your mission statement for w...,English,4.5,"[['Business', 'Money', 'Job Hunting', 'Careers']]",0786882417,0.653846,"{'title_rule': 'first_non_null', 'title_inputs...",NaN,NaN
4,metabooks_127067,group_4,"[metabooks, amazon_books]","History of Modern Art: Painting, Sculpture, Ar...",29.990000,744.0,Prentice Hall,metabooks_127067,h harvard arnason,1986.0,...,18,history of modern art painting sculpture archi...,English,4.2,"[['Arts', 'Photography', 'History', 'Criticism...",0133903605,0.615385,"{'title_rule': 'first_non_null', 'title_inputs...",NaN,NaN


In [11]:
fused_standard_blocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17637 entries, 0 to 17636
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 17637 non-null  object 
 1   _fusion_group_id    17637 non-null  object 
 2   _fusion_sources     17637 non-null  object 
 3   title               17637 non-null  object 
 4   price               16228 non-null  float64
 5   page_count          15714 non-null  float64
 6   publisher           17637 non-null  object 
 7   id                  17637 non-null  object 
 8   author_norm         17637 non-null  object 
 9   publish_year        17630 non-null  float64
 10  author              17637 non-null  object 
 11  rating_number       17637 non-null  int64  
 12  title_norm          17637 non-null  object 
 13  language            17543 non-null  object 
 14  rating              17637 non-null  float64
 15  genres              16766 non-null  object 
 16  isbn

In [12]:
fused_standard_blocker.to_parquet(PIPELINE_DIR / "fused_standard_blocker.parquet")